In [1]:
import MDAnalysis as mda
import mdtraj as md
import os, sys, glob
import numpy as np

from openmm import *
from openmm.app import *
from openmm.unit import *


from chimpss.bridgeport.forcefield import ForceFieldHandler as ff_handler
from chimpss.bridgeport.ligand import Ligand
from chimpss.bridgeport.openmm_joiner import Joiner

from chimpss.motorrow import MotorRow
from datetime import datetime

In [2]:
bridgeport_pdb = '/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.pdb'
bridgeport_xml = '/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.xml'
opm_pdb =        '/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/input_pdb/opm_4ib4.pdb'

In [3]:
# traj = md.load(bridgeport_pdb)
# traj.atom_slice(traj.top.select('protein or resname UNK')).save_pdb('6drx_bp_PL.pdb')
# traj.atom_slice(traj.top.select('protein and not resname UNK')).save_pdb('6drx_bp_P.pdb')
# traj.atom_slice(traj.top.select('resname UNK')).save_pdb('6drx_bp_L.pdb')

In [4]:
def parameterize_from_pdb(pdb_file, nonbonded_method=CutoffNonPeriodic, nonbonded_cutoff=None,
                          lig_resname='UNK', cleanup=True, build_w_obc2=False):

    #Seperate the stuff
    uni = mda.Universe(pdb_file)
    print(len(uni.atoms))
    not_lig = uni.select_atoms(f'not resname {lig_resname}')
    lig = uni.select_atoms('resname UNK')
    not_lig.write('dummy_P.pdb')
    if lig.n_atoms > 0:
        lig.write('dummy_L.pdb')
    else:
        print("Ligand was not found...")

    #Get the Parameters
    ff_xmls = ['amber14/protein.ff14SB.xml',
               'amber14/lipid17.xml',
               '/media/volume/Josephs-Volume/githubs/Bridgeport/ForceFields/wat_opc3.xml'] #That bridgeport directory from the import cells needs to be in path?

    if build_w_obc2:
       ff_xmls.append('implicit/obc2.xml')
    
    ff = ForceField(*ff_xmls)
    pdb = PDBFile('dummy_P.pdb')
    rec_top, rec_pos = pdb.getTopology(), pdb.getPositions()
    
    if nonbonded_cutoff is None:
        rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbonded_method, hydrogenMass=4*amu)
    elif type(nonbonded_cutoff) == int or type(nonbonded_cutoff) == float:
        rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbonded_method, nonbondedCutoff=Quantity(value=nonbonded_cutoff, unit=nanometer), hydrogenMass=4*amu)
    elif type(nonbonded_cutoff) == Quantity:
        rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbonded_method, nonbondedCutoff=nonbonded_cutoff, hydrogenMass=4*amu)
    
    #Ligand Too
    if lig.n_atoms > 0:
        ligand = Ligand(working_dir='./', name='dummy_L', resname='UNK', smiles="CCN(CC)C(=O)N[C@@H]1CN([C@@H]2Cc3c[nH]c4c3c(ccc4)C2=C1)C")
        ligand.prepare_ligand()
        lig_sys, lig_top, lig_pos = ff_handler.ForceFieldHandler('dummy_L.sdf').main(use_pme='NoCutoff')
    
        #Joiner
        sys, top, pos = Joiner((lig_sys, lig_top, lig_pos), (rec_sys, rec_top, rec_pos)).main()
    else:
        sys, top, pos = rec_sys, rec_top, rec_pos
    
    print(sys.getNumParticles())
    for force in sys.getForces():
        if hasattr(force, 'getNonbondedMethod'):
            print(force)
            print(force.getNonbondedMethod(), force.getNumParticles())
            
    if cleanup and lig.n_atoms > 0:
        os.remove('dummy_P.pdb')
        os.remove('dummy_L.pdb')
        os.remove('dummy_L.sdf')
    elif cleanup:
        os.remove('dummy_P.pdb')
    
    return sys, top, pos

In [5]:
def implicit_membrane_force(system, 
                            pore_radius: float=0.75, 
                            soluteDielectric=1, 
                            implicitSolventSaltConc=0.15, 
                            temperature=310,
                            offset=0.009):
    """
    This should be equivalent to Amber ``igb=5``

    The list of parameters to ``addParticle`` is: ``[charge, or, sr]``.

    Parameters
    ----------
    solventDielectric: float
        Dielectric constant for the solvent
    soluteDielectric: float
        Dielectric constant for the solute
    SA: string or None
        Surface area model to use
    cutoff: float or Quantity or None
        Cutoff distance to use. If float, value is in nm. If ``None``,
        then no cutoffs are used.
    kappa: float or Quantity
        Debye kappa parameter related to modelling salt in GB. It has
        units of 1 / length with 1 / nanometer assumed if a float
        is given. A value of zero corresponds to zero salt concentration.
    """
    im_mem = CustomGBForce()

    # Add amber particle parameters
    _ = im_mem.addPerParticleParameter("charge")
    _ = im_mem.addPerParticleParameter("or") # Offset radius
    _ = im_mem.addPerParticleParameter("sr") # Scaled offset radius

    #Add misc. global parameters
    _ = im_mem.addGlobalParameter('epsWater', 78.5) #thickness of membrane slab
    _ = im_mem.addGlobalParameter('soluteDielectric', soluteDielectric) #thickness of membrane slab
    _ = im_mem.addGlobalParameter('offset', offset) #thickness of membrane slab
    _ = im_mem.addGlobalParameter('implicitSolventSaltConc', implicitSolventSaltConc) #thickness of membrane slab
    _ = im_mem.addGlobalParameter('temperature', temperature) #thickness of membrane slab

    # Add pore parameters
    _ = im_mem.addGlobalParameter('poreAlpha', 21.6) 
    _ = im_mem.addGlobalParameter('poreRadius', pore_radius)

    # Descreening Function
    _ = im_mem.addComputedValue("I",  "select(step(r+sr2-or1), 0.5*(1/L-1/U+0.25*(r-sr2^2/r)*(1/(U^2)-1/(L^2))+0.5*log(L/U)/r), 0);"
                                     "U=r+sr2;"
                                     "L=max(or1, D);"
                                     "D=abs(r-sr2)", CustomGBForce.ParticlePairNoExclusions)

    # Born Radius
    _ = im_mem.addComputedValue("B", "1/(1/or-tanh(psi-0.8*psi^2+4.85*psi^3)/radius);"
                                     "psi=I*or; radius=or+offset; offset=0.009", CustomGBForce.SingleParticle)

    # ---- Fig S4 slab function: f(z) = inverse dielectric ----
    # from https://pubs.acs.org/doi/abs/10.1021/acs.jcim.9b00363
    im_mem.addComputedValue("f", "select(step(abs(z*10)-2.5),"
                                 " select(step(abs(z*10)-5.0),"
                                 "  select(step(abs(z*10)-7.5),"
                                 "   select(step(abs(z*10)-12.5),"
                                 "    select(step(abs(z*10)-20.0),"
                                 "     1.25e-2,"
                                 "     (1.66e-3)*(z*10)^2 - (6.72e-2)*abs(z*10) + 6.95e-1),"
                                 "    (3.94e-3)*(z*10)^2 - (1.24e-1)*abs(z*10) + 1.05),"
                                 "   (1.16e-2)*(z*10)^2 - (2.40e-1)*abs(z*10) + 1.48),"
                                 "  (3.44e-2)*(abs(z)*10)^3 - (4.12e-1)*(z*10)^2 + 1.41*abs(z*10) - 4.99e-1),"
                                 " 1.0)", CustomGBForce.SingleParticle)

    # Effective dielectric: eps(z) = 1 / f(z) w/ pore
    im_mem.addComputedValue("eps", "epsWater + (1/f - epsWater)*radialsig;"
                            'radialsig=1/(1+exp(-1*poreAlpha*(abs(sqrt(x^2 + y^2))-poreRadius)))',
                            CustomGBForce.SingleParticle)


    # Add dynamic kappa
    """
    # From https://github.com/openmm/openmm/blob/421eb42a69489cbf0b2b75993102e6cbd25304d9/wrappers/python/openmm/app/amberprmtopfile.py#L287
    # The constant is 1 / sqrt( epsilon_0 * kB / (2 * NA * q^2 * 1000) )
    # where NA is avogadro's number, epsilon_0 is the permittivity of
    # free space, q is the elementary charge (this number matches
    # Amber's kappa conversion factor)
    # Multiply by 0.73 to account for ion exclusions, and multiply by 10
    # to convert to 1/nm from 1/angstroms
    """
    _ = im_mem.addComputedValue('kappa', '7.3 * (50.33355 * sqrt(implicitSolventSaltConc / eps / temperature))', CustomGBForce.SingleParticle)
    
    #Handle Self-Burial / Solvent Exclusion Term
    _ = im_mem.addEnergyTerm("28.3919551*(radius+0.14)^2*(radius/B)^6; radius=or+offset", CustomGBForce.SingleParticle)
    
    #Handle Self-Polar Term
    _ = im_mem.addEnergyTerm("-0.5*138.935485*(1/soluteDielectric-exp(-kappa*B)/eps)*charge^2/B", CustomGBForce.SingleParticle)

    #Handle Particle Pair Polar Term
    _ = im_mem.addEnergyTerm("-138.935485*(1/soluteDielectric-exp(-kappaPair*f)/epsPair)*charge1*charge2/f;"
                             "epsPair = 2 * eps1 * eps2 / (eps1 + eps2);"
                             "kappaPair = 2 * kappa1 * kappa2 / (kappa1 + kappa2);"
                             "f=sqrt(r^2+B1*B2*exp(-r^2/(4*B1*B2)))", CustomGBForce.ParticlePairNoExclusions)

    # Copy charges and radii from NonbondedForce into GBSA
    forces = [force for force in system.getForces()]
    obc2 = [force for force in forces if type(force) == CustomGBForce][0]
    obc2_idx = forces.index(obc2)
    for index in range(obc2.getNumParticles()):
        charge, radius, scaling_factor = obc2.getParticleParameters(index)
        im_mem.addParticle((charge, radius, scaling_factor))
    im_mem.setNonbondedMethod(2)
    
    return im_mem, obc2_idx

In [6]:
def simulate_from_filepair(pdb_top_fn, sys_xml_fn,
                           temp=300, ts=0.001, n_steps=1e5, #100ps
                           dcd_fn='output.dcd', dcd_freq=1000,
                           stdout_fn='output.stdout', stdout_freq=1000,
                           working_dir='./'):
    if not os.path.isdir(working_dir):
        os.makedirs(working_dir, exist_ok=True)
    #Construct
    pdb = PDBFile(pdb_top_fn)
    with open(sys_xml_fn, 'r') as f:
        system = XmlSerializer.deserialize(f.read())
    integrator = LangevinMiddleIntegrator(temp*kelvin, 1/picosecond, ts*picoseconds)
    sim = Simulation(pdb.topology, system, integrator)
    _ = sim.context.setPositions(pdb.positions)
    #Minimize
    _ = sim.minimizeEnergy()
    with open(os.path.join(working_dir,'minimized.pdb'), 'w') as f:
        _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    #Simulate
    _ = sim.reporters.append(DCDReporter(os.path.join(working_dir, dcd_fn), dcd_freq))
    _ = sim.reporters.append(StateDataReporter(os.path.join(working_dir, stdout_fn), stdout_freq, step=True, potentialEnergy=True, temperature=True, speed=True))
    _ = sim.step(n_steps)

    #Final
    state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
    contents = XmlSerializer.serialize(state)
    with open(os.path.join(working_dir, 'final_state.xml'), 'w') as f:
        _ = f.write(contents)
    with open(os.path.join(working_dir,'final_top.pdb'), 'w') as f:
        _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    return None

In [7]:
def assign_force_groups(system):
    """
    Assign each force in the system to a unique group.
    Must be called before creating the Simulation object.
    """
    for i, force in enumerate(system.getForces()):
        force.setForceGroup(i)


def print_full_energy_breakdown(simulation, verbose=False):
    """
    Print energy contributions from all forces, constraints, and barostat.
    Ensures proper evaluation and includes total potential and kinetic energy.
    """
    context = simulation.context
    system = simulation.system

    # Assign each force to its own group (safe even if already done)
    for i, force in enumerate(system.getForces()):
        force.setForceGroup(i)

    # Force OpenMM to compute everything
    context.computeVirtualSites()

    # Get full state
    state_full = context.getState(getEnergy=True, getPositions=True, getVelocities=True)
    total_potential = state_full.getPotentialEnergy()
    kinetic_energy = state_full.getKineticEnergy()
    total_energy = total_potential + kinetic_energy

    my_string = "\n=== Energy Breakdown ===\n"
    my_string += f"Total Potential Energy: {total_potential} \n"
    my_string += f"Kinetic Energy: {kinetic_energy} \n"
    my_string += f"Total Energy: {total_energy} \n"

    # Per-force energy contributions
    force_sum = 0.0 * total_potential.unit
    for i, force in enumerate(system.getForces()):
        state_force = context.getState(getEnergy=True, groups={i})
        energy = state_force.getPotentialEnergy()
        my_string += f"Force {i}: {force.__class__.__name__} -> {energy} \n"
        force_sum += energy

    my_string += f"\nSum of reported forces: {force_sum}\n"
    my_string += f"Difference from total potential: {total_potential - force_sum}\n"
    return my_string

In [8]:
obc_sys, obc_top, obc_pos = parameterize_from_pdb('6drx_bp_P.pdb',
                                                  nonbonded_method=CutoffPeriodic,
                                                  nonbonded_cutoff=Quantity(1.5, nanometer),
                                                  build_w_obc2=True)

5705
Ligand was not found...


/home/exouser/miniconda3/envs/prep/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


5705
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7804088047f0> >
2 5705
<openmm.openmm.CustomGBForce; proxy of <Swig Object of type 'OpenMM::CustomGBForce *' at 0x7804088044f0> >
2 5705


In [9]:
im_mem, obc_idx = implicit_membrane_force(obc_sys,
                                         pore_radius=0.0, 
                                         soluteDielectric=1, 
                                         implicitSolventSaltConc=0.15, 
                                         temperature=300,
                                         offset=0.009)

In [10]:
obc_sys.removeForce(obc_idx)
obc_sys.addForce(im_mem)

5

In [11]:
assign_force_groups(obc_sys)
sim = Simulation(obc_top, obc_sys, LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 3*femtosecond))

In [12]:
working_dir = 'implict_mem_cutoff_1.5_hmr'
if not os.path.isdir(working_dir):
    os.makedirs(working_dir, exist_ok=True)

sim.context.setPositions(obc_pos)
sim.minimizeEnergy()
with open(os.path.join(working_dir,'minimized.pdb'), 'w') as f:
    PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)

#Write the minimized frame to dcd
dcd_filename = os.path.join(working_dir, 'trajectory.dcd')

sim.reporters.append(DCDReporter(os.path.join(working_dir, 'trajectory.dcd'), 5000)) #Every 5k steps
sim.reporters.append(StateDataReporter(os.path.join(working_dir, 'stdout.stdout'), 2500, #Every 2.5k steps
                                       step=True, potentialEnergy=True, temperature=True, speed=True))

In [13]:
from IPython.display import clear_output

In [14]:
sub_sim = 2e4
cache = []
for i in range(50):
    sim.step(sub_sim)
    clear_output(wait=True)
    cache.append((i, print_full_energy_breakdown(sim)))
    if i > 2:
        for j in [-2, -1]:
            print(cache[j][0])
            print(cache[j][1])

48

=== Energy Breakdown ===
Total Potential Energy: 292.98774160659906 kJ/mol 
Kinetic Energy: 21042.611328125 kJ/mol 
Total Energy: 21335.599069731597 kJ/mol 
Force 0: HarmonicBondForce -> 8325.114471435547 kJ/mol 
Force 1: PeriodicTorsionForce -> 19313.828620910645 kJ/mol 
Force 2: NonbondedForce -> -21018.249238831635 kJ/mol 
Force 3: CMMotionRemover -> 0.0 kJ/mol 
Force 4: HarmonicAngleForce -> 11710.77554321289 kJ/mol 
Force 5: CustomGBForce -> -18038.479858398438 kJ/mol 

Sum of reported forces: 292.9895383290095 kJ/mol
Difference from total potential: -0.0017967224104609159 kJ/mol

49

=== Energy Breakdown ===
Total Potential Energy: 2759.939741270906 kJ/mol 
Kinetic Energy: 21000.6328125 kJ/mol 
Total Energy: 23760.572553770908 kJ/mol 
Force 0: HarmonicBondForce -> 8322.446166992188 kJ/mol 
Force 1: PeriodicTorsionForce -> 19252.129608154297 kJ/mol 
Force 2: NonbondedForce -> -21304.30617700302 kJ/mol 
Force 3: CMMotionRemover -> 0.0 kJ/mol 
Force 4: HarmonicAngleForce -> 1181

In [15]:
state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
contents = XmlSerializer.serialize(state)
with open(os.path.join(working_dir, 'final_state.xml'), 'w') as f:
    f.write(contents)
with open(os.path.join(working_dir,'final_top.pdb'), 'w') as f:
    PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
print('Done!')

Done!
